# 38커뮤니케이션 공모주 청약일정 크롤러 (Colab 테스트용)

- 소스: https://www.38.co.kr/html/fund/index.htm?o=k
- 페이지 1~10 크롤링 → pandas DataFrame 확인 → 3섹션 분류 → 엑셀 저장
- Slack 발송은 GitHub Actions에서만 돌리므로 여기선 확인용 출력만

In [ ]:
!pip install -q requests beautifulsoup4 openpyxl

## 1. 크롤링

In [ ]:
import re
from dataclasses import dataclass, asdict
from datetime import date, datetime, timedelta
from typing import Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = 'https://www.38.co.kr'
LIST_URL = f'{BASE_URL}/html/fund/index.htm'
MAX_PAGES = 10
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'ko-KR,ko;q=0.9',
}

@dataclass
class IpoItem:
    name: str
    schedule: str
    start_date: Optional[str]
    end_date: Optional[str]
    fixed_price: str
    desired_price: str
    competition: str
    underwriter: str
    detail_url: str
    analysis_url: str

def parse_schedule(text: str):
    text = text.strip()
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\s*~\s*(?:(\d{4})\.)?(\d{2})\.(\d{2})', text)
    if not m:
        return None, None
    sy, sm, sd, ey, em, ed = m.groups()
    start = f'{sy}-{sm}-{sd}'
    end_year = ey if ey else sy
    if not ey and int(em) < int(sm):
        end_year = str(int(sy) + 1)
    return start, f'{end_year}-{em}-{ed}'

def fetch_page(page: int) -> str:
    r = requests.get(LIST_URL, params={'o': 'k', 'page': page}, headers=HEADERS, timeout=15)
    r.raise_for_status()
    r.encoding = 'euc-kr'
    return r.text

def parse_rows(html: str):
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        a = tr.find('a', href=re.compile(r'/html/fund/\?o=v&no=\d+'))
        if not a:
            continue
        tds = tr.find_all('td')
        if len(tds) < 7:
            continue
        name = a.get_text(strip=True)
        schedule = tds[1].get_text(strip=True)
        start, end = parse_schedule(schedule)
        detail = urljoin(BASE_URL, a.get('href', ''))
        analysis_a = tds[6].find('a')
        analysis = urljoin(BASE_URL, analysis_a.get('href', '')) if analysis_a else ''
        rows.append(IpoItem(
            name=name, schedule=schedule, start_date=start, end_date=end,
            fixed_price=tds[2].get_text(strip=True),
            desired_price=tds[3].get_text(strip=True),
            competition=tds[4].get_text(strip=True),
            underwriter=tds[5].get_text(strip=True),
            detail_url=detail, analysis_url=analysis,
        ))
    return rows

all_items = []
seen = set()
for p in range(1, MAX_PAGES + 1):
    html = fetch_page(p)
    rows = parse_rows(html)
    new = [r for r in rows if r.detail_url not in seen]
    if not new:
        print(f'page {p}: 신규 없음 — 중단')
        break
    seen.update(r.detail_url for r in new)
    all_items.extend(new)
    print(f'page {p}: +{len(new)}건 (누적 {len(all_items)})')

print(f'\n총 {len(all_items)}건')

## 2. DataFrame으로 확인

In [ ]:
df = pd.DataFrame([asdict(x) for x in all_items])
df.head(20)

## 3. 3섹션 분류 (오늘 기준)

In [ ]:
today = date.today()  # 테스트하려면 date(2026, 4, 22)로 고정해도 됨
print(f'기준일: {today}\n')

ongoing, upcoming, recent_end = [], [], []
for it in all_items:
    if not it.start_date or not it.end_date:
        continue
    s = datetime.strptime(it.start_date, '%Y-%m-%d').date()
    e = datetime.strptime(it.end_date, '%Y-%m-%d').date()
    if s <= today <= e:
        ongoing.append(it)
    elif today < s <= today + timedelta(days=7):
        upcoming.append(it)
    elif today - timedelta(days=7) <= e < today:
        recent_end.append(it)

print(f'🔥 공모중 {len(ongoing)}건')
for x in ongoing:
    print(f'  - {x.name} | {x.schedule} | 확정 {x.fixed_price} | 희망 {x.desired_price} | {x.underwriter}')

print(f'\n📅 1주일 내 예정 {len(upcoming)}건')
for x in upcoming:
    print(f'  - {x.name} | {x.schedule} | 희망 {x.desired_price} | {x.underwriter}')

print(f'\n✅ 최근 마감 {len(recent_end)}건')
for x in recent_end:
    print(f'  - {x.name} | {x.schedule} | 확정 {x.fixed_price} | 경쟁률 {x.competition} | {x.underwriter}')

## 4. 엑셀 저장 (다운로드)

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

COLUMNS = [
    ('종목명', 'name', 28), ('공모주일정', 'schedule', 22),
    ('확정공모가', 'fixed_price', 14), ('희망공모가', 'desired_price', 18),
    ('청약경쟁률', 'competition', 14), ('주간사', 'underwriter', 30),
    ('상세 링크', 'detail_url', 55), ('분석 링크', 'analysis_url', 55),
]

wb = Workbook()
ws = wb.active
ws.title = 'IPO 청약일정'

fill = PatternFill(start_color='2F5597', end_color='2F5597', fill_type='solid')
font = Font(color='FFFFFF', bold=True)
for i, (h, _, w) in enumerate(COLUMNS, 1):
    c = ws.cell(1, i, h)
    c.fill = fill; c.font = font
    c.alignment = Alignment(horizontal='center', vertical='center')
    ws.column_dimensions[get_column_letter(i)].width = w

for r, it in enumerate(all_items, 2):
    for i, (_, attr, _) in enumerate(COLUMNS, 1):
        ws.cell(r, i, getattr(it, attr) or '')

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

fname = f'{today.strftime("%Y%m%d")}_38_ipo_schedule.xlsx'
wb.save(fname)
print(f'저장됨: {fname}')

# Colab에서 자동 다운로드
try:
    from google.colab import files
    files.download(fname)
except ImportError:
    pass